# Lab 04.5 — Contours and Object Tracking

## Objectives

- detect the largest object of a selected color;
- compute its contour area;
- draw a bounding box;
- compute its center;
- classify the object's horizontal position as LEFT, CENTER or RIGHT.

This is the first simple tracking experiment.

In [ ]:
import cv2
import numpy as np
from IPython.display import display, Image, clear_output
import time

TARGET = "GREEN"

RANGES = {
    "BLUE":   [((100, 100, 60), (130, 255, 255))],
    "GREEN":  [((35, 80, 60), (85, 255, 255))],
    "YELLOW": [((20, 100, 100), (35, 255, 255))],
    "RED":    [((0, 100, 80), (10, 255, 255)),
               ((170, 100, 80), (179, 255, 255))]
}

In [ ]:
def color_mask(hsv, color_name):
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)

    for lower, upper in RANGES[color_name]:
        mask |= cv2.inRange(hsv, np.array(lower), np.array(upper))

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

def horizontal_region(cx, width):
    if cx < width / 3:
        return "LEFT"
    elif cx < 2 * width / 3:
        return "CENTER"
    else:
        return "RIGHT"

## Track the largest object

Change `TARGET` to RED, GREEN, BLUE or YELLOW.

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

MIN_AREA = 1200

try:
    for _ in range(180):
        ret, frame = cap.read()
        if not ret:
            print("Camera frame not available.")
            break

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = color_mask(hsv, TARGET)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        valid = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]

        if valid:
            cnt = max(valid, key=cv2.contourArea)
            area = cv2.contourArea(cnt)
            x, y, w, h = cv2.boundingRect(cnt)

            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])

                region = horizontal_region(cx, frame.shape[1])

                cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 255, 255), 2)
                cv2.circle(frame, (cx, cy), 6, (255, 255, 255), -1)

                text = f"{TARGET}: {region}  center=({cx},{cy}) area={int(area)}"
                cv2.putText(frame, text, (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65,
                            (255, 255, 255), 2)

        ok, encoded = cv2.imencode(".jpg", frame)
        if ok:
            clear_output(wait=True)
            display(Image(data=encoded.tobytes()))

        time.sleep(0.03)

finally:
    cap.release()
    print("Camera released.")

## Exercise

1. Track a green object.
2. Move it from left to right and verify all three regions.
3. Change the target color.
4. Modify the program to display **LOST** when no valid target is present.
5. Explain why selecting the largest contour is useful in this experiment.